In [ ]:
# 데이터셋 다운
# (원본) https://universe.roboflow.com/site-construction-safety/site-construction-safety

from roboflow import Roboflow
rf = Roboflow(api_key="9xoRm8jQdCrWuNypFp3u")
project = rf.workspace("son-dars0").project("site-construction-safety-14urn")
dataset = project.version(1).download("yolov11")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 85.0%
Version export complete for yolov11 format



Extracting Dataset Version Zip to Site-Construction-Safety-1 in yolov11:: 100%|██████████| 7596/7596 [00:06<00:00, 1092.14it/s]


In [ ]:
# Yolo 파인튜닝
from ultralytics import YOLO
import os

# 모델 로드 (pre trained model)
model = YOLO("yolo11s.pt")

# 모델 학습 실행
results = model.train(
    data="C:\ppe_vision_project\datasets\data.yaml", # yaml
    # epochs=10,              # 학습 반복 횟수
    # epochs=50,              # 학습 반복 횟수
    epochs=100,              # 학습 반복 횟수
    
    # 이마/안경을 헬멧으로 걸러내면, non-helmet 클래스 추가해서 진행해보기
    
    imgsz=640,              # imgsz: Roboflow에서 설정한 640 사이즈 유지
    device=0,               # NVIDIA GPU 사용 시 0, CPU 사용 시 이 줄 삭제
    
    project="PPE_Project",  # 결과 저장 폴더명
    # name="v1_model_10ep",        # 실험 버전 이름
    # name="v1_model_50ep",        # 실험 버전 이름
    name="v1_model_100ep",        # 실험 버전 이름
    plots=True,             # 학습 결과 그래프 자동 생성
    
    save_period=10,
    
    workers=0,
    batch=8,               # 한 번에 공부하는 양
    patience=10,
    amp=False              # 자동 혼합 정밀도 (속도 향상 + 메모리 절약)
)


Ultralytics 8.4.24  Python-3.10.6 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\ppe_vision_project\datasets\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v1_model_100ep, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [5]:
# 평가 수행 (mAP, Precision, Recall 확인)
best_model = YOLO( r"C:\ppe_vision_project\notebooks\runs\detect\PPE_Project\v1_model_100ep\weights\best.pt")

metrics = best_model.val()

print(f"전체 정확도(mAP50): {metrics.box.map50:.3f}")
print(f"헬멧/조끼 등 클래스별 성능: {metrics.box.maps}")

Ultralytics 8.4.24  Python-3.10.6 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 44.35.4 MB/s, size: 46.2 KB)
val: Scanning C:\ppe_vision_project\datasets\valid\labels.cache... 765 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 765/765  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 4.1it/s 11.8s0.2s
                   all        765       5193      0.965      0.945      0.979      0.864
                Helmet        578       1026      0.979      0.925      0.972      0.785
                Person        729       2208      0.959      0.961      0.984      0.912
                  Vest        737       1959      0.956      0.948       0.98      0.895
Speed: 1.4ms preprocess, 9.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to C:\ppe_vision_proj